In [1]:
import cv2 as cv
import numpy as np

In [2]:
INPUT_WIDTH = 640
INPUT_HEIGHT = 640

net = cv.dnn.readNetFromONNX('Model/weights/best.onnx')
net.setPreferableBackend(cv.dnn.DNN_BACKEND_DEFAULT)
net.setPreferableTarget(cv.dnn.DNN_TARGET_CPU)

- prediction [x]
- nms
- results


In [3]:
img = cv.imread('test_images/getty_sample.jpg')

In [ ]:
def get_predictions(image, net):
  image_copy = image.copy()
  r, c, d= image_copy.shape

  max_rc = max(r, c)
  input_image = np.zeros((max_rc, max_rc, 3), dtype=np.uint8)
  input_image[0:r, 0:c] = image

  # convert the image to blob format
  blob = cv.dnn.blobFromImage(input_image, 1/255, (INPUT_HEIGHT, INPUT_WIDTH), swapRB=True, crop=False)
  net.setInput(blob)
  preds = net.forward()
  detections  = preds[0]

  return input_image, detections

def non_max_suppression(input_image, detections):
  boxes = []
  confidences = []

  image_h, image_w = input_image.shape[:2]
  proportion_y = image_h/INPUT_HEIGHT
  proportion_x = image_w/INPUT_WIDTH

  for i in range(len(detections)):
    row = detections[i]
    confidence = row[4]
    if confidence > 0.28:
      class_confidence = row[5]
      if class_confidence > 0.02:
        cx,cy,w,h = row[:4]
        x = int((cx -0.5*w) * proportion_x)
        y = int((cy - 0.5 * h) * proportion_y)
        width = int(w*proportion_x)
        height = int(h*proportion_y)
        box = np.array([x,y,width,height])
        print(box)

        boxes.append(box)
        confidences.append(confidence)
  boxes_np = np.array(boxes).tolist()
  confidences_np = np.array(confidences).tolist()

  index = cv.dnn.NMSBoxes(boxes_np, confidences_np, 0.25, 0.45) # [3]

  return boxes_np, confidences_np, index
  
def results(image, boxes_np, confidences_np, index):
  for ind in index:
    x,y,w,h = boxes_np[ind]
    bb_conf = int(confidences_np[ind] * 100)

    cv.rectangle(image, (x,y), (x+w, y+h), (255,56,0), 3)
    cv.rectangle(image, (x,y-50), (x+w, y), (0,0,0), -1)
    cv.putText(image, f'{bb_conf}%', (x,y-10), cv.FONT_HERSHEY_PLAIN, 2, (255,255,255), 3)


  return image

In [5]:
def yolo_license(img, net):
  input_image, detections = get_predictions(img, net)
  boxes_np, confidences_np, index = non_max_suppression(input_image, detections)
  result = results(img, boxes_np, confidences_np, index)
  return result

In [6]:
result = yolo_license(img, net)
cv.imshow("Image", result)
cv.waitKey(1)
cv.destroyAllWindows()

[955 970 158  46]
[1716 1016  175   46]
[1750 1016  151   46]
[181 961 174  56]
[939 972 153  47]
[953 969 159  47]
[955 970 157  43]
[1699 1011  166   55]
[1709 1013  187   50]
[1743 1013  163   50]
[954 963 160  51]
[1712 1007  183   54]
[171 963 167  58]
[957 966 147  50]
[1713 1010  177   53]


Real time license detection

In [ ]:
capture = cv.VideoCapture('test_images/traffic.mp4')

while True:
  connected, frame = capture.read()
  if not connected:
    break
  
  result = yolo_license(frame, net)
  cv.imshow('Frame', result)
  
  if cv.waitKey(1) & 0xFF == ord('q'):
    break

cv.destroyAllWindows()
capture.release()

[957 100  73  36]
[264 222  82  35]
[1597  265   78   32]
[1595  266   79   32]
[533 360  87  32]
[540 356  81  36]
[2330  390   91   38]
[1968  466   97   39]
[1971  469   90   35]
[2332  389   87   40]
[260 222  89  38]
[1586  266   91   29]
[530 354  99  38]
[2325  389   98   37]
[1968  469   99   39]
[1966  466   96   36]
[961 106  71  33]
[269 216  84  39]
[551 232  91  36]
[1599  263   77   31]
[1595  262   83   33]
[2338  385  100   39]
[1968  467  108   38]
[1967  471  121   41]
[1593  259   86   37]
[2344  385   88   44]
[1970  472  122   45]
[266 217  90  40]
[1588  261   91   32]
[2340  386   99   41]
[1977  470  113   44]
[1967  469  113   40]
[1971  474  122   40]
[212 858 168  61]
[970  99  69  33]
[285 212  86  36]
[1600  266   76   31]
[1597  269   79   30]
[1595  271   73   33]
[554 338 106  37]
[562 335 100  40]
[555 338  96  40]
[2346  394  108   37]
[2350  387   93   43]
[2340  388  106   44]
[1981  466  121   47]
[558 339  93  39]
[2346  393  103   41]
[1981  470  

2025-07-04 09:57:11.265 Python[13571:9389153] +[IMKClient subclass]: chose IMKClient_Modern
2025-07-04 09:57:11.265 Python[13571:9389153] +[IMKInputSession subclass]: chose IMKInputSession_Modern


[302 200  83  37]
[303 201  84  43]
[1592  280   82   35]
[1593  284   78   34]
[605 330  85  35]
[2366  406   99   39]
[2362  406  100   40]
[2009  487  125   43]
[2021  483  120   50]
[2014  483  122   48]
[2026  491  111   41]
[604 325  86  42]
[2368  408   92   42]
[2011  487  120   47]
[2016  485  120   45]
[303 204  82  40]
[597 326 100  38]
[2367  405  100   39]
[2363  409  104   39]
[2018  488  112   44]
[2026  486  106   42]
[2018  488  118   43]
[2022  491  116   41]
[313 200  83  37]
[1595  288   81   34]
[1593  290   80   32]
[598 320  86  35]
[603 315  87  36]
[2368  403  112   45]
[2032  500  117   43]
[2027  496  114   39]
[2028  499  116   38]
[2377  407   98   44]
[2027  498  115   39]
[2027  498  124   44]
[1584  289   95   30]
[595 314 103  38]
[2374  408  106   43]
[2039  501  108   41]
[2029  499  114   39]
[2026  501  125   38]
[323 197  77  36]
[328 204  78  39]
[627 223  76  35]
[1597  294   79   32]
[1598  296   83   32]
[627 315  78  36]
[2371  413  124   43]


: 